# 01 — EDA: Base Studenti

> **Notebook 01 di 7** | Learning Retention Analytics  
> Prima analisi esplorativa: chi sono gli studenti, quali sono i loro esiti e come si presentano i dati?

## Scopo e ambito

Questo notebook è il **primo di due notebook EDA (Exploratory Data Analysis)** del progetto. Esamina la **base studenti**: dati demografici, pattern di iscrizione, distribuzione degli esiti e panoramica dei corsi.

**Cosa copre questo notebook:**
- Dimensioni e scala del dataset
- Distribuzione degli esiti (4 classi e binarizzata)
- Profilo demografico della popolazione studentesca
- Pattern di iscrizione (tempistica di registrazione)
- Panoramica a livello di corso (iscrizioni, completamento, caratteristiche di design)
- Valutazione della qualità dei dati
- Baseline di engagement iniziale (anteprima)

**Cosa viene dopo:**
- **Notebook 02** (`02_eda_engagement_patterns.ipynb`): analisi comportamentale dettagliata — pattern di clickstream giornalieri, segnali di engagement precoce, trend temporali

**Collegamento alle business question:**  
Questa EDA non risponde direttamente a BQ1–BQ5. Piuttosto, stabilisce il profilo della popolazione e la baseline di qualità dei dati da cui dipendono tutte le analisi successive. Pensalo come *comprendere il terreno prima di navigarlo*.

> **Cos'è l'EDA?** L'Exploratory Data Analysis è la pratica di esaminare un dataset prima della modellazione formale o del test di ipotesi. L'obiettivo è scoprire pattern, individuare anomalie, verificare assunzioni e costruire intuizione sui dati. Per un'introduzione approfondita, vedi [`docs/eda-guide.md`](../docs/eda-guide.md).

## Indice

1. [Setup dell'ambiente](#1.-Environment-Setup)
2. [Panoramica del dataset — Forma e scala](#2.-Dataset-Overview-—-Shape-and-Scale)
3. [Distribuzione degli esiti](#3.-Outcome-Distribution)
4. [Esiti per corso](#4.-Outcome-by-Course)
5. [Profilo demografico](#5.-Demographic-Profile)
6. [Pattern di iscrizione](#6.-Enrollment-Patterns)
7. [Panoramica dei corsi](#7.-Course-Landscape)
8. [Valutazione della qualità dei dati](#8.-Data-Quality-Assessment)
9. [Baseline di engagement — Anteprima](#9.-Engagement-Baseline-—-Preview)
10. [Conclusioni e prossimi passi](#10.-Key-Takeaways-and-Next-Steps)

---

**Prerequisiti:**
- La pipeline ETL deve essere stata eseguita: `python -m run_pipeline`
- Il database DuckDB in `data/db/oulad.duckdb` deve contenere tutte e 5 le viste analitiche

**Dataset:** Open University Learning Analytics Dataset (OULAD) — ~32K studenti, 7 corsi, clickstream comportamentale completo. Licenza: CC-BY 4.0.

## 1. Setup dell'ambiente

Configuriamo import, default di visualizzazione e funzioni helper riutilizzabili.

**Note tecniche per il lettore:**
- I notebook risiedono in `notebooks/` ma i moduli del progetto sono in `src/` nella root del progetto. Aggiungiamo la project root a `sys.path` affinché `from src.config import ...` funzioni. La regola del linter `E402` (import non in cima al file) è soppressa per i notebook in `pyproject.toml`.
- Tutte le query al database passano da `src.db.connection.execute_query()` — il layer di astrazione DB del progetto. Questo restituisce un `pandas.DataFrame` e assicura che non si chiami mai `duckdb.connect()` direttamente (vedi [ADR-003](../docs/ADR.md)).
- Le figure vengono salvate in `reports/figures/` a 150 DPI. Poiché `nbstripout` rimuove gli output del notebook prima del commit, i PNG salvati sono il record visivo persistente.

In [ ]:
# --- Path setup ---
# Notebooks live in notebooks/ but project modules are at the project root.
# Adding the root to sys.path lets us import from src/ as if running from root.
# We search upward for pyproject.toml instead of assuming cwd is always notebooks/,
# so the notebook works regardless of where the kernel is launched from
# (JupyterLab, VS Code, Cursor, repo root, etc.).
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    """Walk upward from start until we find pyproject.toml (the repo root marker)."""
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise RuntimeError(
        f"Could not locate project root: no pyproject.toml found in '{start}' "
        "or any parent directory. Run the notebook from within the repository."
    )


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# --- Standard library ---
import logging
import warnings

# --- Third-party ---
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import seaborn as sns

# --- Project modules ---
from src.config import FIGURES_DIR
from src.db.connection import execute_query

# --- Configuration ---
# Suppress noisy warnings in notebook output; errors still surface
warnings.filterwarnings('ignore', category=FutureWarning)
logging.basicConfig(level=logging.WARNING)

# --- Visualization defaults ---
# Consistent style across all project notebooks
sns.set_theme(style='whitegrid', font_scale=1.1)

# Semantic color palette: one color per outcome category
PALETTE_OUTCOME = {
    'Pass': '#4C72B0',        # blue — neutral positive
    'Distinction': '#55A868', # green — strong positive
    'Fail': '#C44E52',        # red — negative
    'Withdrawn': '#8172B3',   # purple — departed (distinct from failed)
}
# Binary version: completed (1) vs not completed (0)
PALETTE_BINARY = {1: '#55A868', 0: '#C44E52'}
LABEL_BINARY = {1: 'Completed', 0: 'Not completed'}
# Label-keyed palette for seaborn when x-axis uses mapped string categories
PALETTE_BINARY_LABELS = {'Completed': '#55A868', 'Not completed': '#C44E52'}
# Sequential palette for heatmaps and continuous scales
PALETTE_SEQUENTIAL = 'YlOrRd'
# Shared axis label — the unit of analysis is the enrollment, not the student
LABEL_NUM_ENROLLMENTS = 'Number of enrollments'

FIG_DPI = 150
FIG_SIZE = (10, 6)
FIG_SIZE_WIDE = (16, 5)

# Ensure figures output directory exists
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(fig, name: str) -> None:
    """Save figure to reports/figures/ with consistent settings."""
    path = FIGURES_DIR / f'{name}.png'
    fig.savefig(path, dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
    print(f'  Saved: {path}')


def query_demo_distribution(column: str) -> pd.DataFrame:
    """Query count and completion rate by a demographic column.

    Returns DataFrame with columns: [category, n, completion_rate].
    Column names are standardized so that plot_demo_completion() works
    regardless of which demographic variable we are analyzing.

    COALESCE handles NULL values (e.g. imd_band) — SQL NULLs become
    pandas NaN which matplotlib cannot use as categorical axis labels.
    Converting to 'Unknown' at the SQL level keeps downstream code clean.
    """
    return execute_query(f'''
        SELECT
            COALESCE(CAST({column} AS VARCHAR), 'Unknown') AS category,
            COUNT(*)                                        AS n,
            ROUND(100.0 * SUM(completed) / COUNT(*), 1)     AS completion_rate
        FROM v_student_enriched
        GROUP BY {column}
        ORDER BY n DESC
    ''')


def plot_demo_completion(
    df: pd.DataFrame,
    title: str,
    figname: str,
    horizontal: bool = False,
) -> None:
    """Bar chart with completion-rate annotations for a demographic variable.

    Parameters
    ----------
    df : DataFrame with columns [category, n, completion_rate]
    title : chart title
    figname : filename without extension (e.g. '01_demo_gender')
    horizontal : if True, horizontal bars (better for long category labels)
    """
    fig, ax = plt.subplots(figsize=FIG_SIZE)

    if horizontal:
        bars = ax.barh(df['category'], df['n'], color='#4C72B0', edgecolor='white')
        for bar, (_, row) in zip(bars, df.iterrows()):
            # Annotate each bar with the completion rate
            ax.text(
                bar.get_width() + ax.get_xlim()[1] * 0.01,
                bar.get_y() + bar.get_height() / 2,
                f"{row['completion_rate']:.1f}% completed",
                va='center', fontsize=10, color='#333333',
            )
        ax.set_xlabel(LABEL_NUM_ENROLLMENTS)
        ax.invert_yaxis()
    else:
        bars = ax.bar(df['category'], df['n'], color='#4C72B0', edgecolor='white')
        for bar, (_, row) in zip(bars, df.iterrows()):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + ax.get_ylim()[1] * 0.01,
                f"{row['completion_rate']:.1f}%",
                ha='center', fontsize=10, color='#333333',
            )
        ax.set_ylabel(LABEL_NUM_ENROLLMENTS)
        plt.xticks(rotation=45, ha='right')

    ax.set_title(title)
    sns.despine()
    fig.tight_layout()
    save_fig(fig, figname)
    plt.show()


# --- Prerequisite check ---
# Verify the database is populated before proceeding
try:
    _check = execute_query('SELECT COUNT(*) AS n FROM v_student_enriched')
    _n_rows = _check['n'].iloc[0]
    if _n_rows == 0:
        raise RuntimeError('v_student_enriched is empty')
    print(f'Database OK — v_student_enriched has {_n_rows:,} rows')
except Exception as exc:
    raise RuntimeError(
        'Cannot query v_student_enriched. '
        "Run 'python -m run_pipeline' first to populate the database."
    ) from exc

## 2. Panoramica del dataset — Forma e scala

Il primo passo in qualsiasi EDA è capire **quanti dati abbiamo** e **come si presentano ad alto livello**.

**Concetto chiave: l'unità di analisi.**  
In OULAD, uno studente può iscriversi a più corsi (moduli). L'unità di analisi è l'**iscrizione studente-modulo** — identificata dalla chiave composita `(id_student, code_module, code_presentation)` — non lo studente singolo.

Questo significa che il numero totale di righe in `v_student_enriched` è il numero di **iscrizioni**, che è maggiore del numero di studenti unici. Uno studente che completa il Corso A ma si ritira dal Corso B appare come due righe: una *completata* e una *non completata*. Entrambe sono punti dati validi per l'analisi di retention.

In [ ]:
# --- Dataset dimensions ---
# How many enrollments, unique students, courses, and presentations?
df_dims = execute_query('''
    SELECT
        COUNT(*)                                                 AS n_enrollments,
        COUNT(DISTINCT id_student)                               AS n_unique_students,
        COUNT(DISTINCT code_module)                              AS n_modules,
        COUNT(DISTINCT code_module || '-' || code_presentation)  AS n_course_presentations
    FROM v_student_enriched
''')

n_enroll = df_dims['n_enrollments'].iloc[0]
n_students = df_dims['n_unique_students'].iloc[0]
n_modules = df_dims['n_modules'].iloc[0]
n_cp = df_dims['n_course_presentations'].iloc[0]

print('=== Dataset Dimensions ===')
print(f'  Total enrollments:       {n_enroll:>8,}')
print(f'  Unique students:         {n_students:>8,}')
print(f'  Modules (courses):       {n_modules:>8}')
print(f'  Course-presentations:    {n_cp:>8}')
print(f'\n  → On average, each student enrolls in {n_enroll / n_students:.1f} modules')

> **Leggere i numeri:** Il conteggio delle iscrizioni è maggiore del conteggio degli studenti unici perché alcuni studenti si iscrivono a più moduli. Questo è normale in un contesto universitario dove gli studenti seguono diversi corsi per semestre.
>
> La distinzione tra *iscrizioni* e *studenti* è rilevante in tutta l'analisi: tutti i tassi di completamento, le suddivisioni demografiche e le metriche di engagement sono calcolati **per iscrizione**, non per persona unica.

In [ ]:
# --- Course summary table ---
# One row per course-presentation with key metrics from v_course_profile.
# We select all columns here because Section 7 (Course Landscape) reuses
# df_courses for scatter plots and correlation analysis.
df_courses = execute_query('''
    SELECT
        code_module,
        code_presentation,
        course_length_days,
        n_enrolled,
        n_completed,
        completion_rate_pct,
        n_withdrew,
        withdrawal_rate_pct,
        n_assessments,
        assessments_per_30_days,
        n_vle_resources,
        n_activity_types
    FROM v_course_profile
    ORDER BY code_module, code_presentation
''')

print(f'=== Course-Presentation Summary ({len(df_courses)} rows) ===\n')
df_courses

> **Cosa ci dice questa tabella:**
> - Ogni riga è un **course-presentation** unico (un modulo specifico offerto in un semestre specifico, es. AAA-2013J).
> - Il **tasso di completamento** varia tra i corsi — alcuni sono notevolmente più alti o più bassi, cosa che BQ4 indagherà.
> - Anche il **design del corso** varia: alcuni corsi hanno più assessment, altri più risorse VLE.
> - La colonna `course_length_days` mostra che i corsi variano da brevi a lunghi — questo influisce sulla tempistica di ritiro (BQ1).
>
> Torneremo a questa tabella nella [Sezione 7 (Panoramica dei corsi)](#7.-Course-Landscape) per l'analisi visiva.

## 3. Distribuzione degli esiti

La **variabile target** in questo progetto è `final_result`, che assume quattro valori:

| Valore | Significato |
|--------|------------|
| **Pass** | Lo studente ha completato il corso e superato l'esame |
| **Distinction** | Lo studente ha completato con lode |
| **Fail** | Lo studente è rimasto iscritto ma non ha superato l'esame |
| **Withdrawn** | Lo studente si è ritirato attivamente prima della fine |

Per la maggior parte delle analisi, **binarizziamo** questo in:
- **Completed** (= Pass + Distinction) → `completed = 1`
- **Not completed** (= Fail + Withdrawn) → `completed = 0`

Questa binarizzazione è documentata in [ADR-006](../docs/ADR.md) e coerente con la letteratura OULAD. La distinzione chiave da tenere a mente: **Withdrawn ≠ Fail**. Gli studenti Withdrawn se ne sono andati attivamente (un segnale comportamentale); gli studenti Fail sono rimasti ma non hanno superato l'esame (un esito accademico). Questa distinzione è rilevante per il design degli interventi (BQ5).

In [ ]:
# --- 4-class outcome distribution ---
df_outcome = execute_query('''
    SELECT
        final_result,
        COUNT(*) AS n_enrollments,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM v_student_enriched
    GROUP BY final_result
    ORDER BY n_enrollments DESC
''')

print('=== Outcome Distribution (4-class) ===\n')
print(df_outcome.to_string(index=False))

# --- Horizontal bar chart ---
# Ordered from largest to smallest category for visual hierarchy
fig, ax = plt.subplots(figsize=FIG_SIZE)
colors = [PALETTE_OUTCOME[r] for r in df_outcome['final_result']]
bars = ax.barh(df_outcome['final_result'], df_outcome['n_enrollments'], color=colors,
               edgecolor='white')

# Annotate each bar with count and percentage
for bar, (_, row) in zip(bars, df_outcome.iterrows()):
    ax.text(
        bar.get_width() + ax.get_xlim()[1] * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{int(row['n_enrollments']):,} ({row['pct']:.1f}%)",
        va='center', fontsize=11,
    )

ax.set_xlabel(LABEL_NUM_ENROLLMENTS)
ax.set_title('How Did Students Finish? (4-class outcome)')
ax.invert_yaxis()  # largest at top
sns.despine(left=True)
fig.tight_layout()
save_fig(fig, '01_outcome_distribution_4class')
plt.show()

In [ ]:
# --- Binary outcome (completed vs not completed) ---
# This is the "headline number" for the project
df_binary = execute_query('''
    SELECT
        completed,
        COUNT(*) AS n,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM v_student_enriched
    GROUP BY completed
    ORDER BY completed DESC
''')

# Donut chart — simple, clean, conveys the key ratio at a glance
fig, ax = plt.subplots(figsize=(7, 7))
labels = [LABEL_BINARY[c] for c in df_binary['completed']]
colors = [PALETTE_BINARY[c] for c in df_binary['completed']]
wedges, texts, autotexts = ax.pie(
    df_binary['n'], labels=labels, colors=colors, autopct='%1.1f%%',
    startangle=90, pctdistance=0.75, textprops={'fontsize': 13},
)
# Create the donut hole
centre_circle = plt.Circle((0, 0), 0.50, fc='white')
ax.add_artist(centre_circle)
ax.set_title('Overall Completion Rate (binary)', fontsize=14, pad=20)
fig.tight_layout()
save_fig(fig, '01_outcome_distribution_binary')
plt.show()

# Print the headline number
completion_rate = df_binary.loc[df_binary['completed'] == 1, 'pct'].iloc[0]
print(f'\nHeadline: {completion_rate:.1f}% of enrollments result in completion.')

> **Interpretazione:**
> - **Withdrawn** è la categoria di non completamento più numerosa. Questi studenti se ne sono andati attivamente — sono il target primario per gli interventi di retention.
> - **Fail** rappresenta studenti che sono rimasti coinvolti ma non hanno superato l'esame. Hanno bisogno di un tipo diverso di supporto (accademico, non motivazionale).
> - Da una prospettiva di piattaforma: una quota significativa degli studenti che si iscrivono **non** completa. Questo è il problema centrale che questo progetto indaga.
>
> Le sezioni successive esploreranno *chi* sono questi studenti e *dove* gli esiti differiscono.

## 4. Esiti per corso

Prima di esaminare i dati demografici o il comportamento, verifichiamo se gli esiti variano sostanzialmente **tra i corsi**. Se un corso ha un tasso di completamento del 70% e un altro del 30%, allora qualsiasi media a livello di popolazione maschera differenze strutturali importanti.

Questa è un'anteprima per **BQ4** ("Come influiscono le caratteristiche del corso sulla retention?"), che il Notebook 06 analizzerà in profondità.

In [ ]:
# --- Outcome proportions by module ---
# 100% stacked bar: normalizes for enrollment size, making courses comparable
df_by_module = execute_query('''
    SELECT
        code_module,
        final_result,
        COUNT(*) AS n
    FROM v_student_enriched
    GROUP BY code_module, final_result
    ORDER BY code_module, final_result
''')

# Pivot to get one column per outcome category
df_pivot = df_by_module.pivot(index='code_module', columns='final_result', values='n').fillna(0)
# Normalize each row to 100%
df_pct = df_pivot.div(df_pivot.sum(axis=1), axis=0) * 100

# Reorder columns for visual consistency (positive outcomes first)
col_order = ['Distinction', 'Pass', 'Fail', 'Withdrawn']
df_pct = df_pct[[c for c in col_order if c in df_pct.columns]]

# Sort modules by completion rate (Distinction + Pass) for easier reading
df_pct['_completion'] = df_pct.get('Distinction', 0) + df_pct.get('Pass', 0)
df_pct = df_pct.sort_values('_completion', ascending=True)
df_pct = df_pct.drop(columns='_completion')

fig, ax = plt.subplots(figsize=FIG_SIZE)
df_pct.plot.barh(
    stacked=True, ax=ax,
    color=[PALETTE_OUTCOME[c] for c in df_pct.columns],
    edgecolor='white', linewidth=0.5,
)
ax.set_xlabel('Percentage of enrollments')
ax.set_title('Outcome Distribution by Module (normalized to 100%)')
ax.legend(title='Outcome', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
sns.despine()
fig.tight_layout()
save_fig(fig, '01_outcome_by_module')
plt.show()

> **Interpretazione:**
> - Gli esiti variano **sostanzialmente** tra i moduli. Questo conferma che i fattori a livello di corso contano e giustifica BQ4.
> - I moduli in fondo al grafico hanno i tassi combinati Pass + Distinction più alti; quelli in cima hanno il maggior numero di ritiri.
> - Il segmento Withdrawn (viola) varia tra i corsi, suggerendo che alcuni design dei corsi possano essere più inclini a innescare abbandoni precoci.
>
> **Implicazione:** Qualsiasi analisi di retention che ignori l'identità del corso rischia il [paradosso di Simpson](https://en.wikipedia.org/wiki/Simpson%27s_paradox) — dove un trend aggregato si inverte all'interno dei sottogruppi. Manterremo la consapevolezza del corso in tutta l'analisi.

## 5. Profilo demografico

Capire **chi sono gli studenti** è essenziale prima di analizzare **cosa hanno fatto**. Questa sezione profila la popolazione studentesca attraverso le principali variabili demografiche ed esamina se i tassi di completamento differiscono per gruppo.

**Variabili esaminate:**
- **Gender** — binario in OULAD (M/F)
- **Age band** — categorizzato come 0-35, 35-55, o 55<=
- **Highest education** — livello di qualifica precedente
- **IMD band** — Index of Multiple Deprivation (indicatore socio-economico, specifico UK)
- **Variabili numeriche** — `num_of_prev_attempts` e `studied_credits`

**Avvertenza importante:** Osservare che un gruppo demografico ha un tasso di completamento più basso **non** significa che quel fattore demografico *causi* un completamento più basso. Correlazione non è causalità. BQ3 confronterà formalmente il potere esplicativo dei dati demografici rispetto al comportamento.

Ogni grafico qui sotto mostra la **distribuzione della popolazione** (altezza della barra = numero di iscrizioni) e il **tasso di completamento** per gruppo (percentuale annotata). Questa doppia vista rivela contemporaneamente sia la dimensione del gruppo che il pattern degli esiti.

### 5a. Genere

In [ ]:
df_gender = query_demo_distribution('gender')
print(df_gender.to_string(index=False))
plot_demo_completion(df_gender, 'Gender Distribution & Completion Rate', '01_demo_gender')

### 5b. Fascia d'età

In [ ]:
df_age = query_demo_distribution('age_band')
print(df_age.to_string(index=False))
plot_demo_completion(df_age, 'Age Band Distribution & Completion Rate', '01_demo_age_band')

### 5c. Titolo di studio più alto

Questa variabile cattura la qualifica più alta che lo studente aveva **prima** di iscriversi. Le categorie vanno da 'No Formal quals' a 'Post Graduate Qualification'. Usiamo barre orizzontali perché le etichette delle categorie sono lunghe.

In [ ]:
df_edu = query_demo_distribution('highest_education')
print(df_edu.to_string(index=False))
plot_demo_completion(
    df_edu, 'Education Level Distribution & Completion Rate',
    '01_demo_education', horizontal=True,
)

### 5d. IMD Band (Index of Multiple Deprivation)

**Cos'è l'IMD?** L'Index of Multiple Deprivation è una misura governativa britannica della deprivazione relativa per piccole aree. Combina indicatori di reddito, occupazione, istruzione, salute, criminalità, alloggio e ambiente. In OULAD:
- **0-10%** = aree più deprivate (decile più basso)
- **90-100%** = aree meno deprivate (decile più alto)

Questo è un proxy socio-economico, non una misura del reddito individuale. Ci dice dell'*area* in cui lo studente vive, non delle sue finanze personali.

> **Nota:** L'IMD band ha valori mancanti noti in OULAD. Li quantificheremo nella sezione Qualità dei dati.

In [ ]:
df_imd = query_demo_distribution('imd_band')
print(df_imd.to_string(index=False))
plot_demo_completion(
    df_imd, 'IMD Band Distribution & Completion Rate',
    '01_demo_imd_band',
)

### 5e. Variabili numeriche: tentativi precedenti e crediti

Due variabili demografiche numeriche meritano attenzione:
- **`num_of_prev_attempts`**: quante volte lo studente ha precedentemente tentato questo modulo (0 = primo tentativo)
- **`studied_credits`**: carico totale di crediti che lo studente sta portando in questa presentazione

Usiamo box plot suddivisi per esito per confrontare le distribuzioni tra i gruppi Completed e Not completed.

In [ ]:
# --- Numeric demographics by outcome ---
df_numeric = execute_query('''
    SELECT
        num_of_prev_attempts,
        studied_credits,
        completed
    FROM v_student_enriched
''')
df_numeric['outcome'] = df_numeric['completed'].map(LABEL_BINARY)

fig, axes = plt.subplots(1, 2, figsize=FIG_SIZE_WIDE)

# Previous attempts
sns.boxplot(
    data=df_numeric, x='outcome', y='num_of_prev_attempts',
    palette=PALETTE_BINARY_LABELS, ax=axes[0],
)
axes[0].set_title('Previous Attempts by Outcome')
axes[0].set_xlabel('')
axes[0].set_ylabel('Number of previous attempts')

# Studied credits
sns.boxplot(
    data=df_numeric, x='outcome', y='studied_credits',
    palette=PALETTE_BINARY_LABELS, ax=axes[1],
)
axes[1].set_title('Studied Credits by Outcome')
axes[1].set_xlabel('')
axes[1].set_ylabel('Total studied credits')

sns.despine()
fig.suptitle('Numeric Demographics by Completion Outcome', fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, '01_demo_numeric_by_outcome')
plt.show()

### Riepilogo demografico

**Osservazioni chiave da questa sezione:**
- I tassi di completamento **variano per gruppo demografico**, ma l'entità delle differenze varia.
- Titolo di studio e IMD band tendono a mostrare la maggiore dispersione nei tassi di completamento.
- Studenti con più tentativi precedenti possono avere tassi di completamento più bassi — ma questo potrebbe riflettere la difficoltà intrinseca del corso piuttosto che la capacità dello studente.

> **Avvertenza sulla causalità:** Osservare che il Gruppo A completa a un tasso più alto del Gruppo B NON significa che *essere nel Gruppo A causi* un completamento più alto. Ci sono fattori confondenti (es. scelta del corso, motivazione, conoscenze pregresse) che non possiamo isolare dalla sola demografica. BQ3 (Notebook 05) confronterà formalmente il potere esplicativo dei dati demografici rispetto al comportamento — la risposta ha implicazioni dirette su dove una piattaforma dovrebbe investire le proprie risorse.

## 6. Pattern di iscrizione

Quando si registrano gli studenti rispetto all'inizio del corso?

In OULAD, `date_registration` è misurato in **giorni relativi all'inizio del corso** (giorno 0). Valori negativi significano che lo studente si è registrato *prima* dell'inizio ufficiale del corso — questo è comune nei sistemi universitari dove le iscrizioni aprono settimane o mesi in anticipo.

**Perché è importante:** La registrazione anticipata può segnalare maggiore motivazione o migliore pianificazione. Se chi si registra presto completa a tassi più alti, questo è un marker comportamentale (anche se non necessariamente causale) che potrebbe informare la tempistica degli interventi.

In [ ]:
# --- Registration day distribution by outcome ---
df_reg = execute_query('''
    SELECT
        date_registration,
        completed
    FROM v_student_enriched
    WHERE date_registration IS NOT NULL
''')
df_reg['outcome'] = df_reg['completed'].map(LABEL_BINARY)

fig, ax = plt.subplots(figsize=FIG_SIZE)
# Overlapping histograms: one per outcome group
for outcome_val, label in LABEL_BINARY.items():
    subset = df_reg[df_reg['completed'] == outcome_val]
    ax.hist(
        subset['date_registration'], bins=50, alpha=0.6,
        label=label, color=PALETTE_BINARY[outcome_val],
    )

# Vertical line at day 0 (course start)
ax.axvline(x=0, color='black', linestyle='--', linewidth=1, label='Course start (day 0)')

ax.set_xlabel('Registration day (relative to course start)')
ax.set_ylabel(LABEL_NUM_ENROLLMENTS)
ax.set_title('When Do Students Register? (by completion outcome)')
ax.legend()
sns.despine()
fig.tight_layout()
save_fig(fig, '01_enrollment_registration_day')
plt.show()

In [ ]:
# --- Registration day summary statistics by outcome ---
# PERCENTILE_CONT is the ANSI SQL standard for computing medians —
# MEDIAN() is a DuckDB shortcut that would break on BigQuery migration
df_reg_stats = execute_query('''
    SELECT
        CASE WHEN completed = 1 THEN 'Completed' ELSE 'Not completed' END AS outcome,
        COUNT(*)                        AS n,
        ROUND(AVG(date_registration), 1)    AS mean_day,
        ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY date_registration), 1) AS median_day,
        MIN(date_registration)              AS min_day,
        MAX(date_registration)              AS max_day
    FROM v_student_enriched
    WHERE date_registration IS NOT NULL
    GROUP BY completed
    ORDER BY completed DESC
''')

print('=== Registration Day Statistics by Outcome ===\n')
df_reg_stats

> **Interpretazione:**
> - La maggior parte degli studenti si registra **prima** dell'inizio del corso (giorno di registrazione negativo), come previsto.
> - Se c'è una differenza tra completers e non-completers nella tempistica di registrazione, ciò suggerisce che la **tempistica di iscrizione** potrebbe essere un segnale precoce debole. Tuttavia, è un dato osservazionale — la registrazione anticipata potrebbe semplicemente essere un proxy di motivazione o supporto istituzionale, non causare direttamente il completamento.
> - Studenti che si registrano molto tardi (giorno di registrazione positivo, cioè dopo l'inizio del corso) potrebbero già essere in svantaggio per via dei contenuti persi.

## 7. Panoramica dei corsi

Passiamo ora da *chi sono gli studenti* a *come sono i corsi*.

Ogni course-presentation ha caratteristiche di design diverse: durata, densità degli assessment, diversità delle risorse VLE. Qualcuna di queste è correlata ai tassi di completamento?

**Avvertenza:** Con solo ~22 course-presentation, la dimensione campionaria è molto piccola per un'analisi di correlazione. I pattern qui sono **suggestivi**, non conclusivi. BQ4 (Notebook 06) analizzerà gli effetti del corso in modo più rigoroso.

In [ ]:
# --- Course design vs. outcomes: 1x3 scatter plot ---
# Each dot is a course-presentation
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Assessment density vs. completion rate
axes[0].scatter(
    df_courses['assessments_per_30_days'], df_courses['completion_rate_pct'],
    s=df_courses['n_enrolled'] / 5,  # bubble size proportional to enrollment
    alpha=0.7, color='#4C72B0', edgecolor='white',
)
axes[0].set_xlabel('Assessments per 30 days')
axes[0].set_ylabel('Completion rate (%)')
axes[0].set_title('Assessment Density vs. Completion')

# 2. VLE resources vs. completion rate
axes[1].scatter(
    df_courses['n_vle_resources'], df_courses['completion_rate_pct'],
    s=df_courses['n_enrolled'] / 5,
    alpha=0.7, color='#55A868', edgecolor='white',
)
axes[1].set_xlabel('Number of VLE resources')
axes[1].set_ylabel('Completion rate (%)')
axes[1].set_title('VLE Resources vs. Completion')

# 3. Course length vs. withdrawal rate
axes[2].scatter(
    df_courses['course_length_days'], df_courses['withdrawal_rate_pct'],
    s=df_courses['n_enrolled'] / 5,
    alpha=0.7, color='#C44E52', edgecolor='white',
)
axes[2].set_xlabel('Course length (days)')
axes[2].set_ylabel('Withdrawal rate (%)')
axes[2].set_title('Course Length vs. Withdrawal')

for ax in axes:
    sns.despine(ax=ax)

fig.suptitle('Course Design Characteristics vs. Student Outcomes\n(bubble size = enrollment)',
             fontsize=13, y=1.04)
fig.tight_layout()
save_fig(fig, '01_course_scatter_outcomes')
plt.show()

In [ ]:
# --- Correlation heatmap of course-level numeric features ---
# Which course characteristics move together?
cols_numeric = [
    'course_length_days', 'n_enrolled', 'completion_rate_pct',
    'withdrawal_rate_pct', 'n_assessments', 'n_vle_resources',
]
# Shorter labels for readability in the heatmap
label_map = {
    'course_length_days': 'Length (days)',
    'n_enrolled': 'Enrolled',
    'completion_rate_pct': 'Completion %',
    'withdrawal_rate_pct': 'Withdrawal %',
    'n_assessments': 'Assessments',
    'n_vle_resources': 'VLE resources',
}

df_corr = df_courses[cols_numeric].rename(columns=label_map).corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    df_corr, annot=True, fmt='.2f', cmap=PALETTE_SEQUENTIAL,
    vmin=-1, vmax=1, center=0, square=True, linewidths=0.5,
    ax=ax,
)
ax.set_title('Correlation Between Course-Level Features')
fig.tight_layout()
save_fig(fig, '01_course_correlation_heatmap')
plt.show()

> **Interpretazione:**
> - Gli scatter plot mostrano se le caratteristiche di design del corso (frequenza degli assessment, quantità di risorse, durata) hanno qualche relazione visibile con gli esiti. La dimensione della bolla codifica il conteggio delle iscrizioni, dando un'idea di quali corsi contribuiscono più dati.
> - La heatmap di correlazione rivela **relazioni lineari** tra le caratteristiche dei corsi. Correlazioni forti (vicine a +1 o -1) suggeriscono caratteristiche che si muovono insieme; correlazioni deboli (vicine a 0) suggeriscono indipendenza.
> - Con solo ~22 data point (course-presentation), **i singoli outlier possono dominare** le correlazioni. Questi risultati sono ipotesi esplorative, non effetti confermati.
>
> BQ4 (Notebook 06) analizzerà gli effetti del corso in dettaglio, controllando per la demografica degli studenti.

## 8. Valutazione della qualità dei dati

**Perché verificare la qualità dei dati?**  
Anche dataset pubblici ben curati possono riservare sorprese: valori mancanti, categorie inattese, outlier o lacune di copertura. Un'EDA professionale include sempre un controllo qualità — *fidarsi ma verificare*.

Cosa verifichiamo:
1. **Valori mancanti** — quali colonne hanno NULL e quanti?
2. **Cardinalità** — quanti valori distinti ha ogni colonna categorica?
3. **Controlli di range** — i valori numerici sono entro i limiti previsti?
4. **Lacune di copertura** — tutte le viste contengono il numero atteso di studenti?

In [ ]:
# --- Missing value audit ---
# Count NULLs per column in v_student_enriched
df_nulls = execute_query('''
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gender IS NULL THEN 1 ELSE 0 END)              AS null_gender,
        SUM(CASE WHEN region IS NULL THEN 1 ELSE 0 END)              AS null_region,
        SUM(CASE WHEN highest_education IS NULL THEN 1 ELSE 0 END)   AS null_education,
        SUM(CASE WHEN imd_band IS NULL THEN 1 ELSE 0 END)            AS null_imd_band,
        SUM(CASE WHEN age_band IS NULL THEN 1 ELSE 0 END)            AS null_age_band,
        SUM(CASE WHEN disability IS NULL THEN 1 ELSE 0 END)          AS null_disability,
        SUM(CASE WHEN date_registration IS NULL THEN 1 ELSE 0 END)   AS null_registration,
        SUM(CASE WHEN final_result IS NULL THEN 1 ELSE 0 END)        AS null_final_result
    FROM v_student_enriched
''')

total = df_nulls['total_rows'].iloc[0]
# Transpose for readability: one row per column
null_cols = [c for c in df_nulls.columns if c.startswith('null_')]
null_summary = pd.DataFrame({
    'column': [c.replace('null_', '') for c in null_cols],
    'null_count': [int(df_nulls[c].iloc[0]) for c in null_cols],
})
null_summary['null_pct'] = (null_summary['null_count'] / total * 100).round(2)

print(f'=== Missing Value Audit ({total:,} total rows) ===\n')
null_summary

In [ ]:
# --- Cardinality check ---
# How many distinct values does each categorical column have?
df_card = execute_query('''
    SELECT
        COUNT(DISTINCT gender)              AS distinct_gender,
        COUNT(DISTINCT region)              AS distinct_region,
        COUNT(DISTINCT highest_education)   AS distinct_education,
        COUNT(DISTINCT imd_band)            AS distinct_imd,
        COUNT(DISTINCT age_band)            AS distinct_age,
        COUNT(DISTINCT disability)          AS distinct_disability,
        COUNT(DISTINCT final_result)        AS distinct_result,
        COUNT(DISTINCT code_module)         AS distinct_module
    FROM v_student_enriched
''')

print('=== Cardinality (distinct values per column) ===\n')
card_summary = df_card.T.reset_index()
card_summary.columns = ['column', 'n_distinct']
card_summary['column'] = card_summary['column'].str.replace('distinct_', '')
card_summary

In [ ]:
# --- Range checks for numeric columns ---
df_ranges = execute_query('''
    SELECT
        MIN(num_of_prev_attempts) AS min_prev_attempts,
        MAX(num_of_prev_attempts) AS max_prev_attempts,
        MIN(studied_credits)      AS min_credits,
        MAX(studied_credits)      AS max_credits,
        MIN(date_registration)    AS min_reg_day,
        MAX(date_registration)    AS max_reg_day,
        MIN(dropout_day)          AS min_dropout_day,
        MAX(dropout_day)          AS max_dropout_day
    FROM v_student_enriched
''')

print('=== Numeric Range Checks ===\n')
# Reshape for readability
ranges = {
    'num_of_prev_attempts': (df_ranges['min_prev_attempts'].iloc[0], df_ranges['max_prev_attempts'].iloc[0]),
    'studied_credits': (df_ranges['min_credits'].iloc[0], df_ranges['max_credits'].iloc[0]),
    'date_registration': (df_ranges['min_reg_day'].iloc[0], df_ranges['max_reg_day'].iloc[0]),
    'dropout_day': (df_ranges['min_dropout_day'].iloc[0], df_ranges['max_dropout_day'].iloc[0]),
}
pd.DataFrame(ranges, index=['min', 'max']).T

In [ ]:
# --- Coverage check: "ghost students" ---
# v_engagement_early only contains enrollments with at least one VLE click
# in days 0-28. Enrollments with ZERO activity are absent from that view.
# The gap between v_student_enriched and v_engagement_early = ghost students.
df_coverage = execute_query('''
    SELECT
        (SELECT COUNT(*) FROM v_student_enriched) AS n_total,
        (SELECT COUNT(*) FROM v_engagement_early) AS n_with_activity
''')

n_total = df_coverage['n_total'].iloc[0]
n_active = df_coverage['n_with_activity'].iloc[0]
n_ghosts = n_total - n_active
ghost_pct = 100 * n_ghosts / n_total

print('=== Engagement Coverage Check ===')
print(f'  Enrollments in v_student_enriched:    {n_total:>8,}')
print(f'  Enrollments in v_engagement_early:    {n_active:>8,}')
print(f'  "Ghost students" (zero activity):     {n_ghosts:>8,} ({ghost_pct:.1f}%)')
print(f'\n  → {n_ghosts:,} enrollments had zero VLE activity in the first 28 days.')
print('    These enrollments are ABSENT from v_engagement_early by design.')
print('    This is not a data quality issue — it is an analytical finding (BQ5 segment).')

> **Verdetto sulla qualità dei dati:**
> - **Valori mancanti**: `imd_band` ha NULL noti (documentati in OULAD). Le altre colonne sono generalmente complete.
> - **Cardinalità**: tutte le colonne categoriche hanno il numero atteso di valori distinti — nessuna categoria inattesa.
> - **Range**: i valori numerici sono entro limiti plausibili. I giorni di registrazione negativi sono previsti (iscrizione pre-corso). I giorni di dropout sono positivi (relativi all'inizio del corso).
> - **Ghost student**: una quota misurabile di iscrizioni ha zero attività VLE nei primi 28 giorni. Questi non sono dati mancanti — questi studenti semplicemente non hanno mai interagito con la piattaforma. Rappresentano un segmento distinto per gli interventi (BQ5).
>
> **Conclusione:** Il dataset è sufficientemente pulito per l'analisi. L'unica avvertenza riguarda i null di `imd_band`, che gestiamo includendo la categoria NULL nelle analisi o escludendola dove indicato.

## 9. Baseline di engagement — Anteprima

Prima di chiudere questo notebook, diamo un **breve sguardo** all'engagement precoce. L'analisi comportamentale completa è nel Notebook 02 — qui stabiliamo solo una baseline.

Usiamo `v_engagement_early`, che aggrega l'attività clickstream per i primi 28 giorni di ciascun corso. **Importante:** questa vista include solo studenti con almeno un click — i "ghost student" identificati nella Sezione 8 non sono in questa vista.

In [ ]:
# --- Early engagement summary + outcome split ---
df_engage = execute_query('''
    SELECT
        ee.total_clicks_first_28,
        ee.active_days_first_28,
        se.completed
    FROM v_engagement_early ee
    JOIN v_student_enriched se
        USING (id_student, code_module, code_presentation)
''')
df_engage['outcome'] = df_engage['completed'].map(LABEL_BINARY)

# Summary statistics
print('=== Early Engagement (first 28 days) ===\n')
print(df_engage.groupby('outcome')[['total_clicks_first_28', 'active_days_first_28']]
      .describe().round(1))

# Violin plot: total clicks by outcome
# Violins show the full distribution shape, not just quartiles
fig, ax = plt.subplots(figsize=FIG_SIZE)
sns.violinplot(
    data=df_engage, x='outcome', y='total_clicks_first_28',
    palette=PALETTE_BINARY_LABELS, inner='quartile', ax=ax,
)
ax.set_xlabel('')
ax.set_ylabel('Total clicks in first 28 days')
ax.set_title('Early Engagement by Completion Outcome')
sns.despine()
fig.tight_layout()
save_fig(fig, '01_engagement_preview_by_outcome')
plt.show()

print('\nNote: "ghost students" (zero activity) are excluded from this view.')
print('The full engagement analysis is in Notebook 02.')

## 10. Conclusioni e prossimi passi

### Cosa abbiamo imparato

1. **Scala**: Il dataset OULAD contiene ~32K iscrizioni di ~28K studenti unici in 22 course-presentation. L'unità di analisi è l'iscrizione, non lo studente.

2. **La sfida del completamento**: Una quota significativa delle iscrizioni non si traduce in completamento. Gli studenti ritirati (che se ne sono andati attivamente) superano in numero quelli che sono rimasti ma non hanno superato l'esame — suggerendo che la **retention**, non il supporto accademico, è la leva principale.

3. **Variazione tra corsi**: I tassi di completamento variano sostanzialmente tra i moduli. Qualsiasi analisi che ignori l'identità del corso rischia il paradosso di Simpson.

4. **La demografica conta — ma quanto?** I tassi di completamento differiscono per titolo di studio, IMD band e altre variabili demografiche. BQ3 testerà se queste differenze sono abbastanza grandi da essere azionabili, o se il comportamento è un segnale più forte.

5. **Segnale di engagement precoce**: Anche in questa breve anteprima, gli studenti che completano mostrano un'attività VLE notevolmente più alta nei primi 28 giorni. BQ2 quantificherà la forza predittiva di questi segnali precoci.

6. **Ghost student**: Un segmento misurabile di studenti si iscrive ma non interagisce mai con il VLE. Sono un target naturale per interventi precoci (BQ5).

7. **Qualità dei dati**: Il dataset è pulito. Le avvertenze note (null di imd_band, copertura ghost student) sono documentate e gestite.

### Cosa viene dopo

| Notebook | Business Question | Focus |
|----------|------------------|-------|
| **02** | — | EDA: pattern di engagement (clickstream giornaliero, trend temporali) |
| **03** | BQ1 | Dove e quando abbandonano gli studenti? |
| **04** | BQ2 | Quali segnali comportamentali precoci predicono l'abbandono? |
| **05** | BQ3 | Demografica vs. comportamento — cosa predice meglio l'esito? |
| **06** | BQ4 | Come influiscono le caratteristiche del corso sulla retention? |
| **07** | BQ5 | Top 3 interventi azionabili |

---

**Riproducibilità:** Tutte le figure sono salvate in `reports/figures/`. Per riprodurre questo notebook, esegui prima `python -m run_pipeline`, poi esegui tutte le celle in ordine.

> **Anteprima del risultato:** Gli studenti che completano hanno un **engagement precoce visibilmente più alto** rispetto ai non-completers — sia nel totale dei click che nella forma della distribuzione (coda più lunga verso l'alto engagement).
>
> Questa è un'**osservazione descrittiva**, non un'affermazione causale. Sarà quantificata con test statistici (t-test, effect size) nel Notebook 04 (BQ2: segnali comportamentali precoci).
>
> Continua al **Notebook 02** (`02_eda_engagement_patterns.ipynb`) per l'analisi comportamentale completa.